# Train Arm A (transformer_standard) on Kaggle - full pipeline test

See `plans/PLAN.md` (question 1.3, Arm A), `phases/phase-2-small-train.md`.

Runs the ACTUAL repo code (`vislm/train.py`, `vislm/backbones/`, the real
`experiments/pillar1_patch_encoder/configs/1_3_arm_A_bpe.yaml`) via the
`nguyennn263/vislm-research-code` dataset attached to this kernel - not a copy pasted
into the notebook. Config overrides (`vislm/args.py`) scale it down to a Kaggle-sized
debug run; the same config + same code run the full job later on the RTX 24GB machine
by passing different overrides.


In [ ]:
import os
print('input:', os.listdir('/kaggle/input'))
for name in os.listdir('/kaggle/input'):
    p = os.path.join('/kaggle/input', name)
    print(p, '->', os.listdir(p) if os.path.isdir(p) else 'file')


In [ ]:
%env PYTHONPATH=/kaggle/input/datasets/nguyennn263/vislm-research-code
!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python /kaggle/input/datasets/nguyennn263/vislm-research-code/setup/download_prepare_data.py \
  --target-gb 0.05 --out-dir /kaggle/working/data/prepared/fineweb2_vi


In [ ]:
!python -m vislm.train /kaggle/input/datasets/nguyennn263/vislm-research-code/pillar1_configs/1_3_arm_A_bpe.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_A_debug \
  train.max_steps=300


In [ ]:
import json

losses = []
with open("/kaggle/working/runs/arm_A_debug/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            losses.append(row["loss"])

summary = {
    "n_steps": len(losses),
    "first_loss": losses[0],
    "last_loss": losses[-1],
    "min_loss": min(losses),
}
print(summary)

with open("/kaggle/working/metrics_train_arm_a_debug.jsonl", "w") as f:
    f.write(json.dumps({"section": "train_arm_a_debug", "results": summary}) + "\n")
